# 🫀 CardioIA — Fase 2 | Parte 2: Classificador de Texto para Triagem Clínica
### Inteligência Artificial no Estetoscópio Digital: TF-IDF & Scikit-Learn

**Autores:** Gabriela de Andrade Alves (RM567740) e Leonardo de Mattos Oliveira (RM568219)  
**Instituição:** FIAP — Inteligência Artificial 2025  
**Metodologia:** Problem Based Learning (PBL)

---

## 🎯 1. Contextualização e Objetivos
A triagem clínica em serviços de emergência (baseada tradicionalmente no Protocolo de Manchester) busca priorizar o atendimento de pacientes com potencial risco de morte iminente.

Nesta etapa, desenvolvemos um classificador automatizado de texto que:
1. Recebe frases médicas simuladas e as rotula em **Alto Risco** (potencial emergência cardiovascular) ou **Baixo Risco** (queixa leve/ambulatorial).
2. Transforma o texto em vetores numéricos através do método **TF-IDF (Term Frequency - Inverse Document Frequency)**.
3. Treina e compara modelos supervisionados do **Scikit-Learn** (**Regressão Logística** e **Árvore de Decisão**).
4. Avalia métricas críticas de saúde (**Recall / Sensibilidade**, Precisão, F1-Score, Acurácia e Matriz de Confusão).
5. Discute vieses algorítmicos, bioética e governança de dados.


## 📦 2. Importações e Configuração de Ambiente


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Estilização gráfica
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
CSV_PATH = os.path.join(DATA_DIR, "frases_risco.csv")

print(f"Caminho da base de dados: {CSV_PATH}")


## 📂 3. Carregamento e Análise Exploratória do Dataset
Carregamos a base com frases já rotuladas e inspecionamos o balanceamento de classes.


In [ ]:
df = pd.read_csv(CSV_PATH)
df['frase'] = df['frase'].str.strip().str.strip('"')
df['situacao'] = df['situacao'].str.strip().str.strip('"').str.lower()
df['rotulo_num'] = df['situacao'].map({'alto risco': 1, 'baixo risco': 0})

print(f"Total de registros: {len(df)}")
print(f"Contagem por classe:\n{df['situacao'].value_counts()}\n")

plt.figure(figsize=(6, 3.5))
sns.countplot(data=df, x='situacao', palette=['#e74c3c', '#2ecc71'])
plt.title("Distribuição das Classes no Dataset de Triagem", fontsize=12, fontweight='bold')
plt.xlabel("Classificação de Risco")
plt.ylabel("Quantidade de Frases")
plt.show()


## 🔢 4. Vetorização de Texto com TF-IDF (Teoria e Prática)
O TF-IDF converte documentos textuais em uma matriz esparsa onde o peso de cada termo reflete sua importância relativa:
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$
- **TF (Term Frequency):** Frequência da palavra $t$ no relato $d$.
- **IDF (Inverse Document Frequency):** $\log \frac{N}{\text{DF}(t)}$, penalizando termos frequentes em todos os textos.


In [ ]:
# Divisão estratificada (75% treino, 25% teste)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['frase'], df['rotulo_num'],
    test_size=0.25,
    random_state=42,
    stratify=df['rotulo_num']
)

# Configuração do TfidfVectorizer
stopwords_pt = [
    'a', 'o', 'e', 'é', 'de', 'do', 'da', 'em', 'um', 'uma', 'para',
    'com', 'no', 'na', 'que', 'os', 'as', 'dos', 'das', 'por', 'ao',
    'se', 'ou', 'mas', 'como', 'já', 'eu', 'ele', 'ela', 'nos', 'não',
    'mais', 'muito', 'há', 'me', 'meu', 'minha', 'seu', 'sua', 'ter',
    'ser', 'ir', 'está', 'estou', 'isso', 'esse', 'essa', 'este',
    'esta', 'aqui', 'ali', 'lá', 'quando', 'onde', 'até', 'depois',
    'antes', 'sobre', 'entre', 'sem', 'também', 'foi', 'são', 'tem',
    'tenho', 'sinto', 'tinha', 'tive', 'às', 'uns', 'umas'
]

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words=stopwords_pt,
    ngram_range=(1, 2),
    min_df=1,
    norm='l2'
)

X_train = tfidf.fit_transform(X_train_raw)
X_test = tfidf.transform(X_test_raw)

print(f"Dimensão da matriz de treino: {X_train.shape} (30 frases x {X_train.shape[1]} termos/n-gramas)")
print(f"Dimensão da matriz de teste:  {X_test.shape} (10 frases x {X_test.shape[1]} termos/n-gramas)")


## 🤖 5. Treinamento dos Modelos de Classificação


In [ ]:
# Modelo 1: Regressão Logística
lr = LogisticRegression(random_state=42, C=1.0)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Modelo 2: Árvore de Decisão
dt = DecisionTreeClassifier(random_state=42, max_depth=4, criterion='gini')
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("Treinamento concluído com sucesso para ambos os modelos!")


## 📊 6. Avaliação e Comparativo de Desempenho Clínico


In [ ]:
def calcular_metricas(y_true, y_pred, nome):
    return {
        "Modelo": nome,
        "Acurácia": accuracy_score(y_true, y_pred),
        "Precisão (Alto Risco)": precision_score(y_true, y_pred, pos_label=1),
        "Recall / Sensibilidade": recall_score(y_true, y_pred, pos_label=1),
        "F1-Score": f1_score(y_true, y_pred, pos_label=1)
    }

tabela_metricas = pd.DataFrame([
    calcular_metricas(y_test, y_pred_lr, "Regressão Logística"),
    calcular_metricas(y_test, y_pred_dt, "Árvore de Decisão")
])

tabela_metricas


### Matrizes de Confusão Gráficas


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
classes = ['Baixo Risco', 'Alto Risco']

sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=classes, yticklabels=classes, annot_kws={'size': 13, 'weight': 'bold'})
axes[0].set_title(f"Regressão Logística (Acc: {accuracy_score(y_test, y_pred_lr):.1%})", fontweight='bold')
axes[0].set_xlabel("Predito")
axes[0].set_ylabel("Real Clínico")

sns.heatmap(confusion_matrix(y_test, y_pred_dt), annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=classes, yticklabels=classes, annot_kws={'size': 13, 'weight': 'bold'})
axes[1].set_title(f"Árvore de Decisão (Acc: {accuracy_score(y_test, y_pred_dt):.1%})", fontweight='bold')
axes[1].set_xlabel("Predito")
axes[1].set_ylabel("Real Clínico")

plt.tight_layout()
plt.show()


## 🔍 7. Análise dos Termos Mais Determinantes (Feature Importance)


In [ ]:
features = np.array(tfidf.get_feature_names_out())
coefs = lr.coef_[0]

top_alto = np.argsort(coefs)[-7:]
top_baixo = np.argsort(coefs)[:7]

termos = np.concatenate([features[top_baixo], features[top_alto]])
valores = np.concatenate([coefs[top_baixo], coefs[top_alto]])
cores = ['#27ae60' if v < 0 else '#e74c3c' for v in valores]

plt.figure(figsize=(10, 5))
plt.barh(termos, valores, color=cores, edgecolor='black', alpha=0.85)
plt.axvline(0, color='gray', linestyle='--')
plt.title("Termos Mais Discriminantes na Triagem (TF-IDF)", fontsize=13, fontweight='bold')
plt.xlabel("Peso do Coeficiente (Verde = Baixo Risco | Vermelho = Alto Risco)")
plt.show()


## 🧪 8. Teste Clínico com Frases Inéditas


In [ ]:
novos_relatos = [
    "Sinto um aperto sufocante no meio do peito irradiando para a mandíbula",
    "Tive uma leve dor nas costas depois de limpar a casa",
    "Acordei com falta de ar desesperadora e coração disparado",
    "Sinto um pouco de dor muscular na panturrilha após caminhar"
]

for relato in novos_relatos:
    v = tfidf.transform([relato])
    pred = lr.predict(v)[0]
    prob = lr.predict_proba(v)[0][1]
    status = "🔴 ALTO RISCO" if pred == 1 else "🟢 BAIXO RISCO"
    print(f"{status} (Prob. Emergência: {prob:.1%})")
    print(f"Relato: "{relato}"\n")


## 🛡️ 9. Análise de Vieses, Bioética e Governança (Requisito PBL)

1. **A Assimetria de Custos em Saúde:**
   Na triagem médica, o custo de um **Falso Negativo** (não identificar um infarto em evolução) é incomensurável comparado ao de um **Falso Positivo** (encaminhar dor muscular leve para avaliação médica). Logo, a métrica clínica mandatória é a **Sensibilidade (Recall)**.

2. **Vieses Demográficos e Sintomas Atípicos:**
   Estudos clínicos da Sociedade Brasileira de Cardiologia mostram que mulheres, idosos e diabéticos frequentemente sofrem de apresentações atípicas de IAM (dispneia súbita, indigestão, náusea e tontura, sem a clássica dor pré-cordial em aperto). Algoritmos treinados exclusivamente em queixas típicas podem subnotificar o risco nesses grupos protegidos.

3. **Arquitetura Human-in-the-Loop:**
   O CardioIA não opera de forma autônoma decisória; ele funciona como um filtro inteligente de priorização de filas hospitalares, garantindo que profissionais de saúde sempre mantenham o controle final sobre a conduta clínica.
